In [2]:
import pandas as pd
import numpy as np
import duckdb
import matplotlib.pyplot as plt
import seaborn as sns

DATA = "data/parcels_shore.parquet"

In [20]:
df = duckdb.sql(f"""
    SELECT
        gis_pin,
        mod_iv_year,
        TRY_CAST(sale_price          AS DOUBLE) AS sale_price,
        TRY_CAST(sale_assessment     AS DOUBLE) AS sale_assessment,
        TRY_CAST(land_value          AS DOUBLE) AS land_value,
        TRY_CAST(improvement_value   AS DOUBLE) AS improvement_value,
        TRY_CAST(net_taxable_value   AS DOUBLE) AS net_taxable_value,
        TRY_CAST(year_constructed    AS INTEGER) AS year_constructed,
        YEAR(TRY_CAST(deed_date_MMDDYY AS DATE)) AS sale_year
    FROM read_parquet('{DATA}')
""").df()

print(f"{len(df):,} rows  |  {df['gis_pin'].nunique():,} unique parcels")
df.head(3)

6,779,800 rows  |  458,611 unique parcels


,gis_pin,mod_iv_year,sale_price,sale_assessment,land_value,improvement_value,net_taxable_value,year_constructed,sale_year
0,0119_70_26,2010,1.0,62400.0,37300.0,49700.0,87000.0,1926,2006
1,0119_71_2,2010,79900.0,57100.0,41000.0,97400.0,128000.0,1920,1995
2,0119_71_4,2010,41900.0,61200.0,31300.0,94000.0,125300.0,1922,1980


In [21]:
# how many sales are lower than $1,000?  (these are likely non-arms-length sales)
duckdb.sql(f"""
    SELECT COUNT(*) AS low_sales_count
    FROM read_parquet('{DATA}')
    WHERE TRY_CAST(sale_price AS DOUBLE) < 1000
""").df()

,low_sales_count
0,2167169


## CPI Adjustment — real 2020 dollars (CPIAUCSL)

In [22]:
cpi_monthly = pd.read_csv("data/CPIAUCSL.csv", parse_dates=["observation_date"])
cpi_annual = (
    cpi_monthly
    .assign(sale_year=cpi_monthly["observation_date"].dt.year)
    .groupby("sale_year")["CPIAUCSL"]
    .mean()
)

CPI_2020 = cpi_annual.loc[2020]
deflator = (CPI_2020 / cpi_annual).rename("deflator")

print(f"Base year CPI (2020): {CPI_2020:.3f}")
print(f"Years covered: {cpi_annual.index.min()} – {cpi_annual.index.max()}")
print(f"\nDeflator range: {deflator.min():.3f}x  ({deflator.idxmin()}) → {deflator.max():.3f}x  ({deflator.idxmax()})")
deflator.tail(10)

Base year CPI (2020): 258.856
Years covered: 1947 – 2026

Deflator range: 0.789x  (2026) → 11.591x  (1947)


sale_year
2017    1.056033
2018    1.030889
2019    1.012529
2020    1.000000
2021    0.955281
2022    0.884595
2023    0.849535
2024    0.825175
2025    0.803995
2026    0.788921
Name: deflator, dtype: float64

In [23]:
df["sale_price_2020"] = df["sale_price"] * df["sale_year"].map(deflator)

n_adjusted = df["sale_price_2020"].notna().sum()
n_missing  = df["sale_price_2020"].isna().sum()
print(f"Adjusted:  {n_adjusted:,}  rows")
print(f"NaN (sale_year outside CPI coverage or null deed date): {n_missing:,}  rows")

df[["gis_pin", "mod_iv_year", "sale_year", "sale_price", "sale_price_2020"]].head(10)

Adjusted:  6,215,499  rows
NaN (sale_year outside CPI coverage or null deed date): 564,301  rows


,gis_pin,mod_iv_year,sale_year,sale_price,sale_price_2020
0,0119_70_26,2010,2006,1.0,1.284272
1,0119_71_2,2010,1995,79900.0,135727.273925
2,0119_71_4,2010,1980,41900.0,131653.521242
3,0119_71_5,2010,1988,45000.0,98486.651871
4,0119_71_7,2010,2006,1.0,1.284272
5,0119_71_10,2010,2004,1.0,1.370272
6,0119_71_12,2010,2006,245000.0,314646.671766
7,0119_71_18,2010,1993,1.0,1.791699
8,0119_71_26,2010,2006,249900.0,320939.605201
9,0119_71_28,2010,1997,76900.0,124005.651300


In [25]:
import os

TMP = DATA + ".tmp.parquet"

duckdb.sql(f"""
    COPY (
        WITH cpi AS (
            SELECT
                YEAR(observation_date)  AS sale_year,
                AVG(CPIAUCSL)           AS cpi_val
            FROM read_csv('data/CPIAUCSL.csv',
                          columns={{'observation_date': 'DATE', 'CPIAUCSL': 'DOUBLE'}})
            GROUP BY 1
        ),
        cpi_2020 AS (SELECT cpi_val FROM cpi WHERE sale_year = 2020)
        SELECT
            p.*,
            YEAR(TRY_CAST(p.deed_date_MMDDYY AS DATE))                          AS sale_year,
            TRY_CAST(p.sale_price AS DOUBLE) * (cpi_2020.cpi_val / cpi.cpi_val) AS sale_price_2020
        FROM read_parquet('{DATA}') p
        CROSS JOIN cpi_2020
        LEFT JOIN cpi ON YEAR(TRY_CAST(p.deed_date_MMDDYY AS DATE)) = cpi.sale_year
    ) TO '{TMP}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

os.replace(TMP, DATA)
print("Done — sale_price_2020 written to", DATA)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Done — sale_price_2020 written to data/parcels_shore.parquet


In [26]:
duckdb.sql(f"""
    SELECT
        COUNT(*)                                                        AS total_rows,
        COUNT(sale_price_2020)                                          AS adjusted,
        COUNT(*) - COUNT(sale_price_2020)                               AS missing,

        -- reason breakdown
        COUNT(*) FILTER (WHERE deed_date_MMDDYY IS NULL
                            OR TRY_CAST(deed_date_MMDDYY AS DATE) IS NULL)  AS bad_deed_date,

        COUNT(*) FILTER (WHERE TRY_CAST(deed_date_MMDDYY AS DATE) IS NOT NULL
                            AND YEAR(TRY_CAST(deed_date_MMDDYY AS DATE))
                                NOT BETWEEN 1947 AND 2026)               AS year_out_of_cpi_range,

        COUNT(*) FILTER (WHERE TRY_CAST(sale_price AS DOUBLE) IS NULL)  AS null_sale_price,

        -- sanity: confirm column exists
        COUNT(sale_price_2020) > 0                                      AS column_present
    FROM read_parquet('{DATA}')
""").df().T

,0
total_rows,6779800
adjusted,6215499
missing,564301
bad_deed_date,549616
year_out_of_cpi_range,14685
null_sale_price,0
column_present,True


In [27]:
duckdb.sql(f"""
    SELECT
        YEAR(TRY_CAST(deed_date_MMDDYY AS DATE)) AS year,
        COUNT(*) AS n_rows
    FROM read_parquet('{DATA}')
    WHERE TRY_CAST(deed_date_MMDDYY AS DATE) IS NOT NULL
      AND YEAR(TRY_CAST(deed_date_MMDDYY AS DATE)) NOT BETWEEN 1947 AND 2026
    GROUP BY 1
    ORDER BY 1
""").df()

,year,n_rows
0,2027,1
1,2031,5
2,2034,16
3,2035,4
4,2039,16
5,2040,2
6,2041,43
7,2042,21
8,2043,2
9,2044,34
